In [1]:
import torch
from PIL import Image
import open_clip
import pandas as pd
import numpy as np

In [2]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', 
                                                             pretrained='pathgenclip_models/pathgenclip_best_breast.pt') 
model.eval()  
tokenizer = open_clip.get_tokenizer('ViT-B-16')

In [10]:
image = preprocess(Image.open("brst_site_image.JPG")).unsqueeze(0)
queries = [
            "lung histopathology image",
            "breast histopathology image",
            "kidney histopathology image"
        ]
text = tokenizer(queries)

with torch.no_grad(), torch.cuda.amp.autocast():
    image_features = model.encode_image(image)
    text_features = model.encode_text(text)
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)

print("Label probs:", text_probs)

C:\Users\Imroze\AppData\Local\Temp\ipykernel_36136\1395705594.py:11: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():


Label probs: tensor([[0.3622, 0.5682, 0.0696]])
